# Prebuilt Agent - `Deep Agent` Detail

우리는 이제부터 Prebuilt Agent Function 인 `create_agent` 기반으로 만들어진 `create_deep_agent` 의   
디테일을 살펴보고 확장 전략에 대해 이해해보겠습니다.

In [ ]:
# create_agent 살펴보기.
from IPython.display import Markdown, display
from langchain.agents.factory import (
    create_agent,
)  # factory 에서 만들어진다는 것을 기억해주세요.

??create_agent

## State Schema 부분 해석
state_schema: An optional `TypedDict` schema that extends `AgentState`.
    When provided, this schema is used instead of `AgentState` as the base  
    schema for merging with middleware state schemas. This allows users to  
    add custom state fields without needing to create custom middleware.  

    Generally, it's recommended to use `state_schema` extensions via middleware  
    to keep relevant extensions scoped to corresponding hooks / tools.  

---

1) `AgentState`를 확장하는 선택적 `TypedDict` 스키마입니다.
2) state_schema 가 입력될 경우, 이 스키마는 미들웨어 상태 스키마와 병합할 때 기본 스키마로 `AgentState` 대신 사용됩니다. 
3) 사용자는 *사용자 정의 미들웨어를 만들 필요 없이* **사용자 정의 상태 필드를 추가할 수 있습니다.**
4) 일반적으로 관련 확장을 해당 훅/도구에 국한시키기 위해서 **미들웨어를 통한 `state_schema` 확장을 사용하는 것이 권장**됩니다.

### State Schema 에 대한 깊은 이해와 Middleware, DeepAgent 에서의 활용 점검
> create_agent 기반으로 만들어졌으니, create_agent 에 대해서 조금 더 파악해보겠습니다.

In [ ]:
# 1. AgentState 가 어떻게 생겼길래, 확장하는 선택적 TypedDict Schema 라고 하는 걸까요?
from langchain.agents.middleware.types import AgentState

print("=" * 50)
??AgentState

2. state_schema 가 입력될 경우, 이 스키마는
**미들웨어 상태 스키마와 병합**할 때 기본 스키마로 `AgentState` 대신 사용

```python
state_schemas: set[type] = {m.state_schema for m in middleware} # 여기가 핵심1
# Use provided state_schema if available, otherwise use base AgentState
base_state = state_schema if state_schema is not None else AgentState
state_schemas.add(base_state) # 여기가 핵심2

resolved_state_schema = _resolve_schema(state_schemas, "StateSchema", None)
input_schema = _resolve_schema(state_schemas, "InputSchema", "input")
output_schema = _resolve_schema(state_schemas, "OutputSchema", "output")

resolved_state_schema # 실제로 쓰이는 State Schema
```

3. 사용자는 *사용자 정의 미들웨어를 만들 필요 없이* **사용자 정의 상태 필드를 추가할 수 있습니다.**

> 미들웨어를 통하지 않고서도 state_schema 에 TypedDict 기반 정의를 해주는 것 만으로도 Field 추가가 가능하다.

4. 일반적으로 관련 확장을 해당 훅/도구에 국한시키기 위해서 **미들웨어를 통한 `state_schema` 확장을 사용하는 것이 권장**됩니다.

> 근데, state_schema Params 말고 Middleware 를 통해 State Schema 를 확장하는 방식으로 사용하길 권장.  
> 이 부분을 DeepAgent 코드와 함께 살펴볼까요?

In [2]:
from deepagents import create_deep_agent

??create_deep_agent

Signature:
create_deep_agent(
    model: str | langchain_core.language_models.chat_models.BaseChatModel | None = None,
    tools: collections.abc.Sequence[langchain_core.tools.base.BaseTool | collections.abc.Callable | dict[str, typing.Any]] | None = None,
    *,
    system_prompt: str | None = None,
    middleware: collections.abc.Sequence[langchain.agents.middleware.types.AgentMiddleware] = (),
    subagents: list[deepagents.middleware.subagents.SubAgent | deepagents.middleware.subagents.CompiledSubAgent] | None = None,
    response_format: Union[langchain.agents.structured_output.ToolStrategy[~SchemaT], langchain.agents.structured_output.ProviderStrategy[~SchemaT], langchain.agents.structured_output.AutoStrategy[~SchemaT], NoneType] = None,
    context_schema: type[typing.Any] | None = None,
    checkpointer: None | bool | langgraph.checkpoint.base.BaseCheckpointSaver = None,
    store: langgraph.store.base.BaseStore | None = None,
    backend: deepagents.backends.protocol.BackendPr

확인해보니, Middleware 말고 State Schema 를 정의하는 부분이 없습니다.  
그렇다면 우리는 Middleware 를 하나하나 살펴볼 필요가 있겠네요.

In [27]:
from deepagents.middleware.subagents import SubAgentMiddleware
from deepagents.middleware.filesystem import FilesystemMiddleware
from deepagents.middleware.patch_tool_calls import PatchToolCallsMiddleware

In [ ]:
?? SubAgentMiddleware

In [ ]:
?? PatchToolCallsMiddleware

In [ ]:
?? FilesystemMiddleware

FileSystem Module 자체를 조금 더 자세히 살펴보겠습니다.

In [ ]:
import deepagents.middleware.filesystem

??deepagents.middleware.filesystem

여기서 발견할 수 있었습니다.  
따라서 create_deep_agent 의 State Field 에 있는 files 키는 기본 AgentState에 원래 없지만,  
FileSystem 미들웨어가 자신만의 상태 스키마를 정의해서 전체 에이전트 상태로 병합됩니다.

그럼 AgentMiddleware 는 어떻게 구성되어 있었을까요?!

In [ ]:
from langchain.agents.middleware.types import AgentMiddleware

??AgentMiddleware